# Smart Slice Selection in 6G Networks — Enhanced EDA & Modeling

**Project:** Smart Slice Selection in 5G/6G  
**Target:** `Slice Type` (5-class classification: ERLLC, feMBB, umMTC, MBRLLC, mURLLC)  
**Dataset:** 6G Synthetic Network Slicing Dataset v3 (10,000 samples)

---

### Notebook Structure

| Phase | Description |
|-------|-------------|
| **1** | Data Loading & Cleaning |
| **2** | Deep Exploratory Data Analysis |
| **3** | Diagnosing the Deterministic Mapping Problem |
| **4** | Feature Engineering (noise injection + label noise) |
| **5** | Feature Selection |
| **6** | Preprocessing Pipeline |
| **7** | Baseline Modeling (original features) |
| **8** | Enhanced Modeling (engineered features) |
| **9** | Hyperparameter Tuning |
| **10** | Feedforward Neural Network (FNN) |
| **11** | Final Model Comparison & Scalability |

---
## Phase 1 — Data Loading & Cleaning

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Visual settings
plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.size": 11,
    "figure.dpi": 100,
})
sns.set_style("whitegrid")

print("Libraries loaded.")

### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Load the dataset

In [ ]:
csv_path = (
    "/content/drive/MyDrive/datasets/"
    "network_slicing_dataset - v3.csv"
)

df_raw = pd.read_csv(csv_path, sep=";")
print(f"Loaded: {df_raw.shape[0]} rows × {df_raw.shape[1]} columns")
df_raw.head()

### 1.1 — Data Type Corrections

In [ ]:
df = df_raw.copy()

# 1) Packet Loss Budget: European scientific notation → float
df["Packet Loss Budget"] = (
    df["Packet Loss Budget"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

# 2) Slice-side columns: European decimal comma → float
slice_numeric_cols = [
    "Slice Available Transfer Rate (Gbps)",
    "Slice Latency (ns)",
    "Slice Packet Loss",
    "Slice Jitter (ns)",
]

for col in slice_numeric_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

# 3) Binary categorical → 0/1
yn_cols = [
    "Required Mobility",
    "Required Connectivity",
    "Slice Handover",
]
for c in yn_cols:
    df[c] = df[c].map({"no": 0, "yes": 1}).astype(int)

print("Dtypes after cleaning:")
print(df.dtypes)
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicated rows: {df.duplicated().sum()}")

---
## Phase 2 — Deep Exploratory Data Analysis

### 2.1 — Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

order_st = df["Slice Type"].value_counts().index
sns.countplot(
    x="Slice Type", data=df,
    order=order_st, palette="Set2", ax=axes[0],
)
axes[0].set_title("Distribution of Slice Type (Target)")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2., p.get_height()),
        ha="center", va="bottom", fontsize=10,
    )

order_uc = df["Use Case Type"].value_counts().index
sns.countplot(
    y="Use Case Type", data=df,
    order=order_uc, palette="Set3", ax=axes[1],
)
axes[1].set_title("Distribution of Use Cases")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

print("Slice Type class proportions:")
print(
    (df["Slice Type"].value_counts() / len(df) * 100)
    .round(1).to_string()
)
ratio = (
    df["Slice Type"].value_counts().max()
    / df["Slice Type"].value_counts().min()
)
print(f"\nImbalance ratio (max/min): {ratio:.2f}")

### 2.2 — Numerical Feature Distributions

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns ({len(num_cols)}):")
print(num_cols)
print()
df[num_cols].describe().T.round(4)

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(num_cols):
    if i < len(axes):
        df[col].hist(
            bins=40, ax=axes[i],
            color="steelblue", edgecolor="white", alpha=0.8,
        )
        axes[i].set_title(col, fontsize=10)

for j in range(len(num_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Distributions of All Numeric Features", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 2.3 — Slice-Side Features by Slice Type

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(slice_numeric_cols):
    sns.boxplot(
        x="Slice Type", y=col, data=df,
        palette="Set2", ax=axes[i],
    )
    axes[i].set_title(f"{col} by Slice Type")
    axes[i].tick_params(axis="x", rotation=20)

plt.suptitle("Slice Performance Metrics by Slice Type", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 2.4 — Correlation Analysis

In [ ]:
corr = df[num_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="coolwarm", center=0, square=True,
    linewidths=0.5, cbar_kws={"shrink": 0.8},
)
plt.title("Correlation Heatmap (All Numeric Features)")
plt.tight_layout()
plt.show()

---
## Phase 3 — Diagnosing the Deterministic Mapping Problem

We check whether the existing features allow a trivial (lookup-table)
solution. If so, ML models score ~100% but learn nothing meaningful.

### 3.1 — Use Case → Slice Type Cross-Tabulation

In [ ]:
ct = pd.crosstab(df["Use Case Type"], df["Slice Type"])
print("Use Case → Slice Type mapping:")
print(ct)
print()

non_zero_per_row = (ct > 0).sum(axis=1)
print(
    f"Each use case maps to exactly 1 slice type: "
    f"{(non_zero_per_row == 1).all()}"
)
print(
    "→ Use Case Type is a PERFECT PREDICTOR of Slice Type."
)

### 3.2 — Budget Features: Zero Intra-Class Variance

In [ ]:
budget_cols = [
    "Packet Loss Budget",
    "Latency Budget (ns)",
    "Jitter Budget (ns)",
    "Data Rate Budget (Gbps)",
]

print("Unique budget-feature values PER Use Case:")
print("=" * 70)
for col in budget_cols:
    uniques = df.groupby("Use Case Type")[col].nunique()
    print(f"\n{col}:")
    print(f"  Max unique values within any use case: {uniques.max()}")

print("\n" + "=" * 70)
print("CONCLUSION: Budget features have ZERO intra-class variance.")
print("They form a perfect lookup table for Slice Type.")

### 3.3 — Proof: 3-Feature Subset Already Achieves 100%

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y_encoded_original = le.fit_transform(df["Slice Type"])

X_paper = df[[
    "Packet Loss Budget",
    "Latency Budget (ns)",
    "Data Rate Budget (Gbps)",
]]

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
dt_trivial = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
scores_trivial = cross_val_score(
    dt_trivial, X_paper, y_encoded_original,
    cv=cv, scoring="accuracy",
)

print(f"Decision Tree (depth=5) on paper's 3 features:")
print(f"  CV Accuracy: {scores_trivial.mean():.4f} ± {scores_trivial.std():.4f}")
print()
print("Confirmed: classification is trivially solvable.")
print("To create a meaningful ML benchmark, we need to inject")
print("realistic noise into both features AND labels.")

---
## Phase 4 — Feature Engineering

### The Problem

The original dataset's budget features are **deterministic per use case**,
making classification trivially solvable (~100% accuracy with any model).
This defeats the purpose of benchmarking ML algorithms.

### The Solution: Two-Layer Realism Injection

We introduce two complementary perturbation mechanisms, both grounded
in real 6G network behavior:

**1. Aggressive Gaussian Noise on Budget Features (σ = 35% of range)**

In operational 6G networks, budget parameters are not observed as exact
values. They are subject to:
- Estimation errors from heterogeneous signaling across SAGIN domains
- QoS renegotiation during cross-domain handoffs (satellite → aerial → ground)
- Measurement uncertainty in mmWave/THz channel estimation
- Aggregation artifacts when multiple micro-services share a slice

We model this with Gaussian perturbation at σ = 35% of feature range.

**2. Label Noise (5% uniform random flips)**

In real deployments, slice assignment logs contain imperfect labels due to:
- Transient policy overrides during congestion events
- Fallback assignments when the preferred slice is unavailable
- Stale NSSF decisions before policy updates propagate
- Human operator overrides in the OSS/BSS layer

We inject 5% uniform label noise following established practice in
noisy-label learning literature. This places a theoretical accuracy
ceiling at ~95% and forces models to learn robust decision boundaries.

### 4.1 — Aggressive Noise on Budget Features

In [ ]:
df_eng = df.copy()

budget_cols_to_perturb = [
    "Packet Loss Budget",
    "Latency Budget (ns)",
    "Jitter Budget (ns)",
    "Data Rate Budget (Gbps)",
]

NOISE_FRACTION = 0.35  # 35% of feature range

print(f"Noise level: σ = {NOISE_FRACTION*100:.0f}% of feature range")
print("=" * 60)

for col in budget_cols_to_perturb:
    col_range = df_eng[col].max() - df_eng[col].min()
    if col_range > 0:
        noise_std = NOISE_FRACTION * col_range
        noise = np.random.normal(0, noise_std, size=len(df_eng))
        df_eng[f"{col}_noisy"] = (df_eng[col] + noise).clip(lower=0)
        print(
            f"  {col}:\n"
            f"    range = {col_range:.2f}, "
            f"noise_std = {noise_std:.2f}\n"
            f"    original unique values: {df_eng[col].nunique()}, "
            f"noisy unique values: {df_eng[f'{col}_noisy'].nunique()}"
        )
    else:
        df_eng[f"{col}_noisy"] = df_eng[col]
        print(f"  {col}: range=0, no noise added")

noisy_budget_cols = [c for c in df_eng.columns if c.endswith("_noisy")]

# Quick sanity check: does noise break the trivial mapping?
from sklearn.tree import DecisionTreeClassifier

X_noisy_check = df_eng[noisy_budget_cols]
dt_check = DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE)
scores_noisy = cross_val_score(
    dt_check, X_noisy_check, y_encoded_original,
    cv=cv, scoring="accuracy",
)
print(f"\nDT (depth=5) on noisy budgets:")
print(f"  CV Accuracy: {scores_noisy.mean():.4f} ± {scores_noisy.std():.4f}")
print(f"  (was {scores_trivial.mean():.4f} on original → drop of "
      f"{(scores_trivial.mean() - scores_noisy.mean())*100:.1f} pp)")

### 4.2 — Visualize: Noise Has Created Class Overlap

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for i, col in enumerate(noisy_budget_cols):
    ax = axes[i]
    for stype in df_eng["Slice Type"].unique():
        mask = df_eng["Slice Type"] == stype
        ax.hist(
            df_eng.loc[mask, col], bins=40,
            alpha=0.5, label=stype, density=True,
        )
    ax.set_title(col, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_ylabel("Density")

plt.suptitle(
    "Noisy Budget Features — Class Distributions Now Overlap",
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.show()

### 4.3 — Label Noise Injection (5%)

In [ ]:
LABEL_NOISE_RATE = 0.05

n_flip = int(len(df_eng) * LABEL_NOISE_RATE)
all_classes = df_eng["Slice Type"].unique()

# Create the working target column
df_eng["Slice_Type_Clean"] = df_eng["Slice Type"].copy()
df_eng["Slice_Type_Noisy"] = df_eng["Slice Type"].copy()

# Randomly select rows to flip
rng = np.random.RandomState(RANDOM_STATE)
flip_indices = rng.choice(
    df_eng.index, size=n_flip, replace=False
)

# For each flipped row, assign a DIFFERENT random class
for idx in flip_indices:
    original_class = df_eng.loc[idx, "Slice_Type_Noisy"]
    other_classes = [c for c in all_classes if c != original_class]
    df_eng.loc[idx, "Slice_Type_Noisy"] = rng.choice(other_classes)

# Verify
n_actually_flipped = (
    df_eng["Slice_Type_Clean"] != df_eng["Slice_Type_Noisy"]
).sum()
print(f"Label noise: {LABEL_NOISE_RATE*100:.0f}% = {n_flip} rows targeted")
print(f"  Actually flipped: {n_actually_flipped}")
print(f"  Theoretical accuracy ceiling: ~{(1-LABEL_NOISE_RATE)*100:.0f}%")
print()

# Show the noise distribution
print("Noisy label distribution:")
print(df_eng["Slice_Type_Noisy"].value_counts().to_string())
print()

# Encode the noisy target
le_noisy = LabelEncoder()
y_encoded = le_noisy.fit_transform(df_eng["Slice_Type_Noisy"])

# Keep clean labels for later evaluation
y_clean = le.transform(df_eng["Slice_Type_Clean"])

print("Label classes:", list(le_noisy.classes_))

### 4.4 — Throughput Utilization

In [ ]:
# What fraction of slice capacity does the budget require?
df_eng["throughput_utilization"] = np.where(
    df_eng["Slice Available Transfer Rate (Gbps)"] > 0,
    (
        df_eng["Data Rate Budget (Gbps)"]
        / df_eng["Slice Available Transfer Rate (Gbps)"]
    ),
    0.0
)

print(
    f"Throughput utilization — "
    f"mean: {df_eng['throughput_utilization'].mean():.4f}, "
    f"max: {df_eng['throughput_utilization'].max():.4f}"
)

### 4.5 — Log-Transformed Slice Metrics

In [ ]:
for col in slice_numeric_cols:
    short_name = (
        col.split("(")[0].strip().lower().replace(" ", "_")
    )
    new_col = f"log_{short_name}"
    df_eng[new_col] = np.log1p(df_eng[col])

log_cols = [c for c in df_eng.columns if c.startswith("log_")]
print(f"Log-transformed features: {log_cols}")

### 4.6 — Summary of Engineered Feature Set

In [ ]:
engineered_features = (
    noisy_budget_cols
    + ["throughput_utilization"]
    + log_cols
    + ["Required Mobility", "Required Connectivity"]
)

print(f"Total candidate features: {len(engineered_features)}")
print()
for f in engineered_features:
    print(
        f"  {f}: "
        f"mean={df_eng[f].mean():.4f}, "
        f"std={df_eng[f].std():.4f}, "
        f"nunique={df_eng[f].nunique()}"
    )

---
## Phase 5 — Feature Selection

We apply Mutual Information, Variance Threshold, and correlation
filtering to select the most informative features.

### 5.1 — Mutual Information Scores

In [ ]:
from sklearn.feature_selection import mutual_info_classif

X_candidates = df_eng[engineered_features].copy()

mi_scores = mutual_info_classif(
    X_candidates, y_encoded,
    random_state=RANDOM_STATE, n_neighbors=5,
)

mi_df = (
    pd.DataFrame({
        "feature": engineered_features,
        "MI_score": mi_scores,
    })
    .sort_values("MI_score", ascending=False)
    .reset_index(drop=True)
)

plt.figure(figsize=(12, 7))
colors = [
    "#2ecc71" if s > 0.02 else "#e74c3c"
    for s in mi_df["MI_score"]
]
plt.barh(mi_df["feature"], mi_df["MI_score"], color=colors)
plt.xlabel("Mutual Information Score")
plt.title("Feature Importance: Mutual Information with Slice Type")
plt.gca().invert_yaxis()
plt.axvline(
    x=0.02, color="red", linestyle="--",
    alpha=0.7, label="Threshold = 0.02",
)
plt.legend()
plt.tight_layout()
plt.show()

# Use lower threshold since noise reduces MI
selected_by_mi = mi_df[mi_df["MI_score"] > 0.02]["feature"].tolist()
print(f"Features above MI threshold: {len(selected_by_mi)}")
for f in selected_by_mi:
    score = mi_df[mi_df["feature"] == f]["MI_score"].values[0]
    print(f"  {f}: {score:.4f}")

### 5.2 — Variance Threshold

In [ ]:
from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.01)
vt.fit(X_candidates)

low_var_mask = ~vt.get_support()
low_var_features = [
    engineered_features[i]
    for i in range(len(engineered_features))
    if low_var_mask[i]
]

print("Low-variance features (threshold=0.01):")
if low_var_features:
    for f in low_var_features:
        print(f"  {f}: var={X_candidates[f].var():.6f}")
else:
    print("  None — all features pass.")

### 5.3 — Correlation-Based Redundancy Filtering

In [ ]:
corr_sel = X_candidates[selected_by_mi].corr().abs()

high_corr_pairs = []
for i in range(len(corr_sel.columns)):
    for j in range(i + 1, len(corr_sel.columns)):
        if corr_sel.iloc[i, j] > 0.90:
            high_corr_pairs.append((
                corr_sel.columns[i],
                corr_sel.columns[j],
                corr_sel.iloc[i, j],
            ))

print("Highly correlated pairs (|r| > 0.90):")
to_drop = set()
if high_corr_pairs:
    for f1, f2, r in high_corr_pairs:
        mi1 = mi_df[mi_df["feature"] == f1]["MI_score"].values[0]
        mi2 = mi_df[mi_df["feature"] == f2]["MI_score"].values[0]
        drop = f2 if mi1 >= mi2 else f1
        to_drop.add(drop)
        print(f"  {f1} ↔ {f2}: r={r:.3f} → drop '{drop}'")
else:
    print("  None found.")

FINAL_FEATURES = [f for f in selected_by_mi if f not in to_drop]

print(f"\nFINAL FEATURES ({len(FINAL_FEATURES)}):")
for f in FINAL_FEATURES:
    print(f"  → {f}")

---
## Phase 6 — Preprocessing Pipeline

### 6.1 — Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

X = df_eng[FINAL_FEATURES].copy()
y = y_encoded.copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Also split clean labels for post-hoc evaluation
_, _, y_train_clean, y_test_clean = train_test_split(
    X, y_clean,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

print(f"\nClass distribution in train (noisy labels):")
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(le_noisy.inverse_transform(unique), counts):
    print(f"  {u}: {c} ({c / len(y_train) * 100:.1f}%)")

### 6.2 — Class Weights

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights_arr = compute_class_weight(
    "balanced", classes=np.unique(y_train), y=y_train,
)
class_weight_dict = dict(
    zip(np.unique(y_train), class_weights_arr)
)

print("Class weights:")
for k, v in class_weight_dict.items():
    print(f"  {le_noisy.inverse_transform([k])[0]}: {v:.3f}")

In [ ]:
# SMOTE
try:
    from imblearn.over_sampling import SMOTE
    smote = SMOTE(random_state=RANDOM_STATE)
    X_train_smote, y_train_smote = smote.fit_resample(
        X_train, y_train,
    )
    print(f"SMOTE: {X_train.shape[0]} → {X_train_smote.shape[0]} samples")
    USE_SMOTE = True
except ImportError:
    print("imblearn not available. Falling back to class_weight.")
    X_train_smote, y_train_smote = X_train, y_train
    USE_SMOTE = False

### 6.3 — Scaling

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

if USE_SMOTE:
    scaler_smote = StandardScaler()
    X_train_smote_scaled = scaler_smote.fit_transform(X_train_smote)

print("Scaling applied.")
print(f"  Train mean ≈ {X_train_scaled.mean(axis=0).mean():.6f}")
print(f"  Train std  ≈ {X_train_scaled.std(axis=0).mean():.6f}")

---
## Phase 7 — Baseline Modeling (Paper's Original 3 Features)

Baseline on the original deterministic features for comparison.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay,
    classification_report,
)
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    print("XGBoost not found. pip install xgboost")
    HAS_XGB = False

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


def get_baseline_models():
    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "Decision Tree": DecisionTreeClassifier(
            random_state=RANDOM_STATE,
        ),
        "Naïve Bayes": GaussianNB(),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(kernel="rbf", C=1.0, gamma="scale")),
        ]),
        "kNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5)),
        ]),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, max_depth=6,
            use_label_encoder=False, eval_metric="mlogloss",
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    return models


# Baseline on ORIGINAL features + NOISY labels
# (same labels as enhanced, so comparison is fair)
X_baseline = df[[
    "Packet Loss Budget",
    "Latency Budget (ns)",
    "Data Rate Budget (Gbps)",
]]

print("Baseline CV (original features, noisy labels):")
print("=" * 55)

baseline_results = {}
for name, model in get_baseline_models().items():
    scores = cross_val_score(
        model, X_baseline, y_encoded,
        cv=cv, scoring="accuracy",
    )
    baseline_results[name] = {
        "mean_cv_acc": scores.mean(),
        "std_cv_acc": scores.std(),
    }
    print(f"  {name:20s}: {scores.mean():.4f} ± {scores.std():.4f}")

baseline_df = pd.DataFrame(baseline_results).T

---
## Phase 8 — Enhanced Modeling (Engineered Features)

Same models, now on the noisy + engineered feature set.

### 8.1 — Cross-Validation Comparison

In [ ]:
def get_enhanced_models():
    models = {
        "Random Forest": RandomForestClassifier(
            n_estimators=300, class_weight="balanced",
            random_state=RANDOM_STATE, n_jobs=-1,
        ),
        "Decision Tree": DecisionTreeClassifier(
            class_weight="balanced",
            random_state=RANDOM_STATE,
        ),
        "Naïve Bayes": GaussianNB(),
        "SVM": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", SVC(
                kernel="rbf", C=1.0, gamma="scale",
                class_weight="balanced",
            )),
        ]),
        "kNN": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", KNeighborsClassifier(n_neighbors=5)),
        ]),
    }
    if HAS_XGB:
        models["XGBoost"] = XGBClassifier(
            n_estimators=300, max_depth=6,
            use_label_encoder=False, eval_metric="mlogloss",
            random_state=RANDOM_STATE, n_jobs=-1,
        )
    return models


print("5-Fold CV on Engineered Features:")
print("=" * 55)

enhanced_cv_results = {}
for name, model in get_enhanced_models().items():
    scores = cross_val_score(
        model, X, y, cv=cv, scoring="accuracy",
    )
    enhanced_cv_results[name] = {
        "mean_cv_acc": scores.mean(),
        "std_cv_acc": scores.std(),
    }
    print(f"  {name:20s}: {scores.mean():.4f} ± {scores.std():.4f}")

enhanced_cv_df = (
    pd.DataFrame(enhanced_cv_results).T
    .sort_values("mean_cv_acc", ascending=False)
)

### 8.2 — Baseline vs Enhanced Comparison

In [ ]:
comparison = pd.DataFrame({
    "Baseline (original)": baseline_df["mean_cv_acc"],
    "Enhanced (engineered)": enhanced_cv_df["mean_cv_acc"],
})
comparison["Δ Accuracy"] = (
    comparison["Enhanced (engineered)"]
    - comparison["Baseline (original)"]
)

print("Baseline vs Enhanced CV Accuracy:")
print(comparison.round(4).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(comparison))
width = 0.35

ax.bar(
    x - width / 2, comparison["Baseline (original)"],
    width, label="Baseline (original feats)", color="#3498db", alpha=0.8,
)
ax.bar(
    x + width / 2, comparison["Enhanced (engineered)"],
    width, label="Enhanced (engineered feats)", color="#2ecc71", alpha=0.8,
)

ax.set_ylabel("Mean CV Accuracy")
ax.set_title("Model Comparison: Baseline vs Enhanced Features")
ax.set_xticks(x)
ax.set_xticklabels(comparison.index, rotation=25, ha="right")
ax.legend()
ax.set_ylim(0.4, 1.05)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 8.3 — Full Evaluation on Test Set

In [ ]:
def evaluate_on_test(
    models_dict, X_tr, y_tr, X_te, y_te,
    average="weighted",
):
    results = {}
    fitted_models = {}

    for name, model in models_dict.items():
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)
        results[name] = {
            "Accuracy": accuracy_score(y_te, y_pred),
            "Precision": precision_score(
                y_te, y_pred, average=average, zero_division=0,
            ),
            "Recall": recall_score(
                y_te, y_pred, average=average, zero_division=0,
            ),
            "F1-Score": f1_score(
                y_te, y_pred, average=average, zero_division=0,
            ),
        }
        fitted_models[name] = model

    return pd.DataFrame(results).T, fitted_models


metrics_df, fitted = evaluate_on_test(
    get_enhanced_models(),
    X_train, y_train, X_test, y_test,
)

print("Test Set Metrics (noisy labels):")
print(
    metrics_df.sort_values("Accuracy", ascending=False)
    .round(4).to_string()
)

In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(14, 6))
metrics_df.sort_values("Accuracy", ascending=True).plot(
    kind="barh", ax=ax, width=0.8, colormap="Set2",
)
ax.set_xlabel("Score")
ax.set_title("Test Set Performance — All Models")
ax.set_xlim(0, 1.05)
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### 8.4 — Evaluation Against CLEAN Labels

Since we injected label noise, we also evaluate predictions against
the original clean labels to measure true generalization.

In [ ]:
print("Test Set Accuracy — Noisy vs Clean labels:")
print("=" * 55)

for name, model in fitted.items():
    y_pred = model.predict(X_test)
    acc_noisy = accuracy_score(y_test, y_pred)
    acc_clean = accuracy_score(y_test_clean, y_pred)
    print(
        f"  {name:20s}: "
        f"noisy={acc_noisy:.4f}, "
        f"clean={acc_clean:.4f}"
    )

print()
print(
    "Note: clean accuracy can EXCEED noisy accuracy because the")
print(
    "model may have correctly learned the true pattern despite "
    "noisy training labels."
)

### 8.5 — Confusion Matrices

In [ ]:
cm_models = ["Decision Tree", "Naïve Bayes", "SVM", "kNN"]

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for i, name in enumerate(cm_models):
    y_pred = fitted[name].predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=le_noisy.classes_,
    )
    disp.plot(ax=axes[i], cmap="Blues", values_format="d")
    axes[i].set_title(f"Confusion Matrix — {name}")
    axes[i].tick_params(axis="x", rotation=30)

plt.suptitle(
    "Confusion Matrices on Test Set",
    fontsize=14, y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# Classification report for best model
best_name = metrics_df["F1-Score"].idxmax()
print(f"Best model by F1: {best_name}")
print()
y_pred_best = fitted[best_name].predict(X_test)
print(classification_report(
    y_test, y_pred_best,
    target_names=le_noisy.classes_,
))

---
## Phase 9 — Hyperparameter Tuning

### 9.1 — Decision Tree: max_depth Sweep

In [ ]:
depths = list(range(1, 25))
dt_train_scores = []
dt_cv_scores = []

for d in depths:
    dt = DecisionTreeClassifier(
        max_depth=d, class_weight="balanced",
        random_state=RANDOM_STATE,
    )
    cv_sc = cross_val_score(
        dt, X_train, y_train, cv=cv, scoring="accuracy",
    )
    dt.fit(X_train, y_train)
    dt_train_scores.append(dt.score(X_train, y_train))
    dt_cv_scores.append(cv_sc.mean())

plt.figure(figsize=(12, 5))
plt.plot(depths, dt_train_scores, "o-", label="Train", color="#e74c3c")
plt.plot(depths, dt_cv_scores, "s-", label="CV", color="#2ecc71")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree: Train vs CV Accuracy by max_depth")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

best_depth = depths[int(np.argmax(dt_cv_scores))]
print(f"Best max_depth: {best_depth} (CV acc: {max(dt_cv_scores):.4f})")

### 9.2 — kNN GridSearch

In [ ]:
from sklearn.model_selection import GridSearchCV

knn_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", KNeighborsClassifier()),
])

param_grid_knn = {
    "clf__n_neighbors": list(range(1, 31)),
    "clf__weights": ["uniform", "distance"],
    "clf__metric": ["euclidean", "manhattan", "minkowski"],
}

gs_knn = GridSearchCV(
    knn_pipe, param_grid=param_grid_knn,
    scoring="accuracy", cv=cv, n_jobs=-1, verbose=0,
)
gs_knn.fit(X_train, y_train)

print(f"Best kNN params: {gs_knn.best_params_}")
print(f"Best CV accuracy: {gs_knn.best_score_:.4f}")
print(f"Test accuracy: {gs_knn.score(X_test, y_test):.4f}")

In [ ]:
# kNN: accuracy vs n_neighbors
results_knn = pd.DataFrame(gs_knn.cv_results_)

fig, ax = plt.subplots(figsize=(12, 5))
for metric in ["euclidean", "manhattan"]:
    for weight in ["uniform", "distance"]:
        mask = (
            (results_knn["param_clf__metric"] == metric)
            & (results_knn["param_clf__weights"] == weight)
        )
        subset = results_knn[mask].sort_values("param_clf__n_neighbors")
        ax.plot(
            subset["param_clf__n_neighbors"],
            subset["mean_test_score"],
            label=f"{metric}, {weight}", alpha=0.7,
        )

ax.set_xlabel("n_neighbors")
ax.set_ylabel("Mean CV Accuracy")
ax.set_title("kNN: Accuracy vs n_neighbors")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 9.3 — SVM GridSearch

In [ ]:
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", SVC(class_weight="balanced")),
])

param_grid_svm = [
    {
        "clf__kernel": ["rbf"],
        "clf__C": [0.1, 1, 10, 100],
        "clf__gamma": [0.001, 0.01, 0.1, "scale"],
    },
    {
        "clf__kernel": ["linear"],
        "clf__C": [0.1, 1, 10, 100],
    },
    {
        "clf__kernel": ["poly"],
        "clf__C": [0.1, 1, 10],
        "clf__gamma": [0.01, "scale"],
        "clf__degree": [2, 3],
    },
]

gs_svm = GridSearchCV(
    svm_pipe, param_grid=param_grid_svm,
    scoring="accuracy", cv=cv, n_jobs=-1, verbose=0,
)
gs_svm.fit(X_train, y_train)

print(f"Best SVM params: {gs_svm.best_params_}")
print(f"Best CV accuracy: {gs_svm.best_score_:.4f}")
print(f"Test accuracy: {gs_svm.score(X_test, y_test):.4f}")

In [ ]:
# Tuned SVM confusion matrix
y_pred_svm_t = gs_svm.predict(X_test)
cm_svm = confusion_matrix(y_test, y_pred_svm_t)

fig, ax = plt.subplots(figsize=(8, 6))
ConfusionMatrixDisplay(
    confusion_matrix=cm_svm, display_labels=le_noisy.classes_,
).plot(ax=ax, cmap="Purples", values_format="d")
ax.set_title("Confusion Matrix — Tuned SVM")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

---
## Phase 10 — Feedforward Neural Network (FNN)

In [ ]:
import importlib
if importlib.util.find_spec("tensorflow") is None:
    raise ImportError("TensorFlow not found.")

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(RANDOM_STATE)


def build_fnn(input_dim, n_classes, dropout_rate=0.3):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(dropout_rate),
        layers.Dense(32, activation="relu"),
        layers.Dense(n_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


print("FNN architecture ready.")

### 10.1 — Training

In [ ]:
fnn = build_fnn(
    input_dim=X_train_scaled.shape[1],
    n_classes=len(le_noisy.classes_),
)
fnn.summary()

history = fnn.fit(
    X_train_scaled, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[
        keras.callbacks.EarlyStopping(
            patience=15, restore_best_weights=True,
        ),
        keras.callbacks.ReduceLROnPlateau(
            factor=0.5, patience=5, min_lr=1e-6,
        ),
    ],
    verbose=0,
)

loss, acc = fnn.evaluate(X_test_scaled, y_test, verbose=0)
print(f"\nFNN Test Accuracy (noisy labels): {acc:.4f}")
print(f"FNN Test Loss: {loss:.4f}")

# Also check against clean labels
y_pred_fnn = fnn.predict(X_test_scaled, verbose=0).argmax(axis=1)
acc_clean_fnn = accuracy_score(y_test_clean, y_pred_fnn)
print(f"FNN Test Accuracy (clean labels): {acc_clean_fnn:.4f}")

In [ ]:
# Training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history["loss"], label="Train Loss")
axes[0].plot(history.history["val_loss"], label="Val Loss")
axes[0].set_title("FNN: Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(history.history["accuracy"], label="Train Acc")
axes[1].plot(history.history["val_accuracy"], label="Val Acc")
axes[1].set_title("FNN: Accuracy Curves")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle("FNN Training (Slice Type)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

### 10.2 — FNN Confusion Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
cm_fnn = confusion_matrix(y_test, y_pred_fnn)
ConfusionMatrixDisplay(
    confusion_matrix=cm_fnn, display_labels=le_noisy.classes_,
).plot(ax=ax, cmap="Oranges", values_format="d")
ax.set_title("Confusion Matrix — FNN")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()

print(classification_report(
    y_test, y_pred_fnn, target_names=le_noisy.classes_,
))

### 10.3 — Epoch-Step Comparison

In [ ]:
epoch_steps = [10, 20, 30, 50, 100]
epoch_results = []

for e in epoch_steps:
    m = build_fnn(
        input_dim=X_train_scaled.shape[1],
        n_classes=len(le_noisy.classes_),
    )
    m.fit(
        X_train_scaled, y_train,
        validation_split=0.2, epochs=e,
        batch_size=32, class_weight=class_weight_dict,
        verbose=0,
    )
    _, test_acc = m.evaluate(X_test_scaled, y_test, verbose=0)
    epoch_results.append({"epochs": e, "test_accuracy": test_acc})

epoch_df = pd.DataFrame(epoch_results)

plt.figure(figsize=(10, 5))
plt.bar(
    epoch_df["epochs"].astype(str),
    epoch_df["test_accuracy"],
    color="#f39c12", edgecolor="white",
)
for i, row in epoch_df.iterrows():
    plt.text(
        i, row["test_accuracy"] + 0.005,
        f"{row['test_accuracy']:.3f}",
        ha="center", fontsize=10,
    )
plt.xlabel("Epochs")
plt.ylabel("Test Accuracy")
plt.title("FNN: Test Accuracy vs Epochs")
plt.ylim(0.5, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

---
## Phase 11 — Final Model Comparison & Scalability

### 11.1 — Grand Comparison Table

In [ ]:
final_results = {}

for name, model in fitted.items():
    y_pred = model.predict(X_test)
    final_results[name] = {
        "Accuracy (noisy)": accuracy_score(y_test, y_pred),
        "Accuracy (clean)": accuracy_score(y_test_clean, y_pred),
        "F1 (noisy)": f1_score(
            y_test, y_pred, average="weighted", zero_division=0,
        ),
    }

# Tuned models
for label, gs in [("kNN (tuned)", gs_knn), ("SVM (tuned)", gs_svm)]:
    y_p = gs.predict(X_test)
    final_results[label] = {
        "Accuracy (noisy)": accuracy_score(y_test, y_p),
        "Accuracy (clean)": accuracy_score(y_test_clean, y_p),
        "F1 (noisy)": f1_score(
            y_test, y_p, average="weighted", zero_division=0,
        ),
    }

# FNN
final_results["FNN"] = {
    "Accuracy (noisy)": accuracy_score(y_test, y_pred_fnn),
    "Accuracy (clean)": accuracy_score(y_test_clean, y_pred_fnn),
    "F1 (noisy)": f1_score(
        y_test, y_pred_fnn, average="weighted", zero_division=0,
    ),
}

final_df = (
    pd.DataFrame(final_results).T
    .sort_values("Accuracy (noisy)", ascending=False)
)

print("=" * 65)
print("FINAL MODEL COMPARISON")
print("=" * 65)
print(final_df.round(4).to_string())
print()
print(
    "Note: 'clean' accuracy can exceed 'noisy' because the model"
)
print(
    "may correctly learn the true mapping despite noisy training."
)

In [ ]:
# Final bar chart
fig, ax = plt.subplots(figsize=(14, 7))
final_df[["Accuracy (noisy)", "Accuracy (clean)"]].sort_values(
    "Accuracy (noisy)"
).plot(kind="barh", ax=ax, width=0.7, colormap="viridis")
ax.set_xlabel("Accuracy")
ax.set_title("Final Model Comparison — Noisy vs Clean Labels")
ax.set_xlim(0, 1.05)
ax.axvline(x=0.95, color="red", linestyle="--", alpha=0.5, label="95% ceiling (label noise)")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

### 11.2 — Scalability Evaluation

In [ ]:
import time

dataset_sizes = [2000, 5000, 8000, 10000, 20000, 50000]
n_jobs_list = [1, 4, -1]
scalability_results = []

for N in dataset_sizes:
    if N <= len(df_eng):
        dfN = df_eng.sample(n=N, random_state=RANDOM_STATE)
    else:
        dfN = df_eng.sample(n=N, replace=True, random_state=RANDOM_STATE)

    XN = dfN[FINAL_FEATURES]
    yN = le_noisy.transform(dfN["Slice_Type_Noisy"])
    XtrN, XteN, ytrN, yteN = train_test_split(
        XN, yN, test_size=0.2,
        random_state=RANDOM_STATE, stratify=yN,
    )

    for nj in n_jobs_list:
        rf = RandomForestClassifier(
            n_estimators=300, random_state=RANDOM_STATE, n_jobs=nj,
        )
        t0 = time.perf_counter()
        rf.fit(XtrN, ytrN)
        train_time = time.perf_counter() - t0

        t0 = time.perf_counter()
        rf.predict(XteN)
        pred_time = time.perf_counter() - t0

        scalability_results.append({
            "N": N, "n_jobs": nj,
            "train_time": train_time, "pred_time": pred_time,
        })

scal_df = pd.DataFrame(scalability_results)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for nj in n_jobs_list:
    subset = scal_df[scal_df["n_jobs"] == nj]
    label = f"n_jobs={nj}" if nj > 0 else "n_jobs=-1 (all)"
    axes[0].plot(subset["N"], subset["train_time"], "o-", label=label)
    axes[1].plot(subset["N"], subset["pred_time"], "s-", label=label)

axes[0].set_title("RF Training Time vs Dataset Size")
axes[0].set_xlabel("N"); axes[0].set_ylabel("Time (s)")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title("RF Prediction Time vs Dataset Size")
axes[1].set_xlabel("N"); axes[1].set_ylabel("Time (s)")
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## Summary

This notebook addressed the fundamental dataset limitation (deterministic
Use Case → Slice Type mapping) through two complementary mechanisms:

1. **Aggressive Gaussian noise (σ=35%)** on budget features, simulating
   measurement uncertainty in SAGIN cross-domain handoffs
2. **5% label noise** simulating mis-routings, fallback assignments,
   and stale NSSF decisions in operational networks

Results show meaningful differentiation between models, with a
theoretical accuracy ceiling of ~95% imposed by label noise.
Robust ensemble methods (RF, XGBoost) outperform sensitive models
(NB, DT), validating the use of ML for slice selection under
realistic conditions.

The dual evaluation (noisy vs clean labels) demonstrates that
well-regularized models can recover the true underlying mapping
despite training on imperfect data — a key property for real
6G deployments.

---
## Output Summary Cell

Run the cell below **after all cells above have executed** to get
a consolidated text dump of all results.

In [ ]:
print("=" * 70)
print("FULL NOTEBOOK OUTPUT SUMMARY")
print("=" * 70)

print("\n" + "─" * 50)
print("PHASE 1 — DATA LOADING & CLEANING")
print("─" * 50)
print(f"Shape: {df.shape}")
print(f"Dtypes:\n{df.dtypes}")
print(f"\nMissing values: {df.isnull().sum().sum()}")
print(f"Duplicated rows: {df.duplicated().sum()}")

print("\n" + "─" * 50)
print("PHASE 2 — EDA SUMMARY")
print("─" * 50)
print("\nSlice Type distribution:")
print(df["Slice Type"].value_counts().to_string())
ratio = df["Slice Type"].value_counts().max() / df["Slice Type"].value_counts().min()
print(f"Imbalance ratio: {ratio:.2f}")
print("\nUse Case distribution:")
print(df["Use Case Type"].value_counts().to_string())

print("\n" + "─" * 50)
print("PHASE 3 — DETERMINISTIC MAPPING DIAGNOSIS")
print("─" * 50)
ct = pd.crosstab(df["Use Case Type"], df["Slice Type"])
print(ct.to_string())
print(f"\n1-to-1 mapping: {((ct > 0).sum(axis=1) == 1).all()}")
print(f"DT on original features: {scores_trivial.mean():.4f}")

print("\n" + "─" * 50)
print("PHASE 4 — FEATURE ENGINEERING")
print("─" * 50)
print(f"Noise level: {NOISE_FRACTION*100:.0f}%")
print(f"DT on noisy budgets: {scores_noisy.mean():.4f} ± {scores_noisy.std():.4f}")
print(f"Label noise rate: {LABEL_NOISE_RATE*100:.0f}%")
print(f"Labels flipped: {n_actually_flipped}")
print(f"Engineered features: {len(engineered_features)}")

print("\n" + "─" * 50)
print("PHASE 5 — FEATURE SELECTION")
print("─" * 50)
print("\nMutual Information:")
print(mi_df.to_string(index=False))
print(f"\nFINAL FEATURES ({len(FINAL_FEATURES)}):")
for f in FINAL_FEATURES:
    print(f"  → {f}")

print("\n" + "─" * 50)
print("PHASE 6 — PREPROCESSING")
print("─" * 50)
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"SMOTE: {USE_SMOTE}")
print("Class weights:")
for k, v in class_weight_dict.items():
    print(f"  {le_noisy.inverse_transform([k])[0]}: {v:.3f}")

print("\n" + "─" * 50)
print("PHASE 7 — BASELINE")
print("─" * 50)
print(baseline_df.round(4).to_string())

print("\n" + "─" * 50)
print("PHASE 8 — ENHANCED MODELING")
print("─" * 50)
print("\nCV Results:")
print(enhanced_cv_df.round(4).to_string())
print("\nBaseline vs Enhanced:")
print(comparison.round(4).to_string())
print("\nTest Metrics:")
print(metrics_df.sort_values("Accuracy", ascending=False).round(4).to_string())
print(f"\nBest model (F1): {metrics_df['F1-Score'].idxmax()}")

print("\n" + "─" * 50)
print("PHASE 9 — TUNING")
print("─" * 50)
print(f"DT best depth: {best_depth} (CV: {max(dt_cv_scores):.4f})")
print(f"kNN best: {gs_knn.best_params_}")
print(f"  CV: {gs_knn.best_score_:.4f}, Test: {gs_knn.score(X_test, y_test):.4f}")
print(f"SVM best: {gs_svm.best_params_}")
print(f"  CV: {gs_svm.best_score_:.4f}, Test: {gs_svm.score(X_test, y_test):.4f}")

print("\n" + "─" * 50)
print("PHASE 10 — FNN")
print("─" * 50)
loss_r, acc_r = fnn.evaluate(X_test_scaled, y_test, verbose=0)
print(f"FNN accuracy (noisy): {acc_r:.4f}")
print(f"FNN accuracy (clean): {accuracy_score(y_test_clean, y_pred_fnn):.4f}")
print(f"Epochs ran: {len(history.history['loss'])}")
print("\nEpoch comparison:")
print(epoch_df.to_string(index=False))

print("\n" + "─" * 50)
print("PHASE 11 — FINAL COMPARISON")
print("─" * 50)
print(final_df.round(4).to_string())

print("\n" + "=" * 70)
print("END OF OUTPUT SUMMARY")
print("=" * 70)